# Bank Customer Churn — Model Training

Teaching notebook for interns: trains a small binary-classification pipeline
(scikit-learn `Pipeline` + `ColumnTransformer`) on a 6-feature subset of the
public **Bank Customer Churn ("Churn_Modelling")** dataset, then saves the
same three artifacts used by the `BankChurnDeployment` API/Streamlit project:

- `artifacts/bank_churn_model.joblib`
- `artifacts/model_metadata.json`
- `artifacts/feature_schema.json`

This mirrors the pattern used in the original `Attrition_Prediction_Completed`
notebook, just with a smaller feature set (6 features instead of 47).

In [ ]:
!pip show joblib

: 

In [1]:
# Imports
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
TARGET_COLUMN = "Exited"
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = Path("data/bank_churn_train.csv")


## 1. Load data

Source: Kaggle "Churn Modelling" dataset (bank customer churn), trimmed down
to 6 predictive features plus the binary target `Exited`.

Original columns dropped to satisfy the 6-feature limit: `RowNumber`,
`CustomerId`, `Surname`, `Tenure`, `NumOfProducts`, `HasCrCard`,
`EstimatedSalary`.

In [2]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


(10000, 7)


,CreditScore,Geography,Gender,Age,Balance,IsActiveMember,Exited
0,619,France,Female,42,0.00,1,1
1,608,Spain,Female,41,83807.86,1,0
2,502,France,Female,42,159660.80,0,1
3,699,France,Female,39,0.00,0,0
4,850,Spain,Female,43,125510.82,1,0


In [3]:
print(df.isna().sum())
print()
print(df[TARGET_COLUMN].value_counts(normalize=True))


CreditScore       0
Geography         0
Gender            0
Age               0
Balance           0
IsActiveMember    0
Exited            0
dtype: int64

Exited
0    0.7963
1    0.2037
Name: proportion, dtype: float64


## 2. Prepare the modelling dataset

In [4]:
if TARGET_COLUMN not in df.columns:
    raise KeyError(f"Target column '{TARGET_COLUMN}' is missing.")

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN].astype(int)

if y.nunique() != 2:
    raise ValueError(f"{TARGET_COLUMN} must contain exactly two classes; found {sorted(y.unique())}.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE,
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"Numeric features: {numeric_features}")
print(f"Categorical features: {categorical_features}")


Training rows: 8000
Test rows: 2000
Numeric features: ['CreditScore', 'Age', 'Balance', 'IsActiveMember']
Categorical features: ['Geography', 'Gender']


## 3. Preprocessing + candidate models

In [5]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
], remainder="drop")

models = {
    "logistic_regression": LogisticRegression(
        max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE,
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=400, class_weight="balanced", min_samples_leaf=2,
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
    "gradient_boosting": GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.05, max_depth=2, random_state=RANDOM_STATE,
    ),
}

pipelines = {
    name: Pipeline(steps=[("preprocessor", clone(preprocessor)), ("model", model)])
    for name, model in models.items()
}


## 4. Cross-validate and pick the best model

In [6]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
scoring = ["average_precision", "roc_auc"]

leaderboard_rows = []
cv_results_by_model = {}

for name, pipeline in pipelines.items():
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_results_by_model[name] = scores
    leaderboard_rows.append({
        "model": name,
        "cv_average_precision_mean": scores["test_average_precision"].mean(),
        "cv_average_precision_std": scores["test_average_precision"].std(),
        "cv_roc_auc_mean": scores["test_roc_auc"].mean(),
    })

leaderboard = pd.DataFrame(leaderboard_rows).sort_values(
    "cv_average_precision_mean", ascending=False
).reset_index(drop=True)
leaderboard.to_csv(ARTIFACT_DIR / "model_leaderboard.csv", index=False)
leaderboard


,model,cv_average_precision_mean,cv_average_precision_std,cv_roc_auc_mean
0,gradient_boosting,0.583767,0.019421,0.800107
1,random_forest,0.548912,0.012798,0.775216
2,logistic_regression,0.452180,0.011020,0.766341


In [7]:
best_model_name = leaderboard.loc[0, "model"]
best_model_metrics = leaderboard.loc[0].to_dict()
print("Selected model:", best_model_name)
best_model_metrics


Selected model: gradient_boosting


{'model': 'gradient_boosting',
 'cv_average_precision_mean': 0.5837671605907825,
 'cv_average_precision_std': 0.01942119513894369,
 'cv_roc_auc_mean': 0.8001065086190167}

## 5. Evaluate on the holdout set

In [8]:
holdout_pipeline = clone(pipelines[best_model_name])
holdout_pipeline.fit(X_train, y_train)

probabilities = holdout_pipeline.predict_proba(X_test)[:, 1]
predictions = (probabilities >= 0.5).astype(int)

holdout_metrics = {
    "accuracy": accuracy_score(y_test, predictions),
    "precision": precision_score(y_test, predictions),
    "recall": recall_score(y_test, predictions),
    "f1": f1_score(y_test, predictions),
    "roc_auc": roc_auc_score(y_test, probabilities),
    "average_precision": average_precision_score(y_test, probabilities),
}

print(classification_report(y_test, predictions))
print(confusion_matrix(y_test, predictions))
holdout_metrics


              precision    recall  f1-score   support

           0       0.85      0.97      0.90      1593
           1       0.72      0.32      0.44       407

    accuracy                           0.84      2000
   macro avg       0.78      0.64      0.67      2000
weighted avg       0.82      0.84      0.81      2000

[[1542   51]
 [ 277  130]]


{'accuracy': 0.836,
 'precision': 0.7182320441988951,
 'recall': 0.3194103194103194,
 'f1': 0.4421768707482993,
 'roc_auc': 0.8100735558362677,
 'average_precision': 0.5870904996398868}

## 6. Refit on all data and save artifacts

The pipeline is refit on the full labelled dataset (after honest holdout
evaluation above) before being saved for deployment.

In [9]:
final_pipeline = clone(pipelines[best_model_name])
final_pipeline.fit(X, y)

model_path = ARTIFACT_DIR / "bank_churn_model.joblib"
metadata_path = ARTIFACT_DIR / "model_metadata.json"
schema_path = ARTIFACT_DIR / "feature_schema.json"

joblib.dump(final_pipeline, model_path)

feature_schema = {
    "feature_columns": X.columns.tolist(),
    "numeric_columns": numeric_features,
    "categorical_columns": categorical_features,
    "categorical_values_seen": {
        column: sorted(X[column].dropna().unique().tolist())
        for column in categorical_features
    },
    "target_column": TARGET_COLUMN,
    "excluded_identifier_columns": ["RowNumber", "CustomerId", "Surname"],
}

metadata = {
    "best_model": best_model_name,
    "model_version": "1.0",
    "decision_threshold": 0.5,
    "training_rows": int(len(X)),
    "positive_rows": int(y.sum()),
    "negative_rows": int((y == 0).sum()),
    "validation": "3-fold stratified cross-validation",
    "selection_metric": "average_precision",
    "best_model_metrics": {
        key: (float(value) if isinstance(value, (int, float, np.floating)) else value)
        for key, value in best_model_metrics.items()
    },
    "holdout_metrics": {key: float(value) for key, value in holdout_metrics.items()},
    "source_dataset": "Kaggle: Churn Modelling (bank customer churn), 6-feature subset",
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

with open(schema_path, "w", encoding="utf-8") as file:
    json.dump(feature_schema, file, indent=2)

print(f"Saved model: {model_path.resolve()}")
print(f"Saved metadata: {metadata_path.resolve()}")
print(f"Saved feature schema: {schema_path.resolve()}")


Saved model: /home/claude/work/newproj/BankChurnDeployment/artifacts/bank_churn_model.joblib
Saved metadata: /home/claude/work/newproj/BankChurnDeployment/artifacts/model_metadata.json
Saved feature schema: /home/claude/work/newproj/BankChurnDeployment/artifacts/feature_schema.json


## 7. Sanity check: load artifacts back and predict one row

In [10]:
loaded_model = joblib.load(model_path)
loaded_schema = json.loads(schema_path.read_text())
loaded_metadata = json.loads(metadata_path.read_text())

sample = X.iloc[[0]][loaded_schema["feature_columns"]]
probability = float(loaded_model.predict_proba(sample)[0, 1])
threshold = float(loaded_metadata["decision_threshold"])

print("Sample features:")
display(sample)
print(f"Churn probability: {probability:.4f}")
print(f"Predicted class (threshold={threshold}):", int(probability >= threshold))


Sample features:


,CreditScore,Geography,Gender,Age,Balance,IsActiveMember
0,619,France,Female,42,0.0,1


Churn probability: 0.1636
Predicted class (threshold=0.5): 0


## 8. Export a small test CSV for the batch-prediction demo

Held out from the training refit conceptually (in practice: a random sample
of feature columns + true labels), used later by the Streamlit batch page
and FastAPI examples.

In [11]:
test_sample = df.sample(n=50, random_state=RANDOM_STATE).reset_index(drop=True)
test_sample.insert(0, "Customer_Ref", [f"CUST-{i+1:04d}" for i in range(len(test_sample))])
test_sample.to_csv("data/test_bank_churn.csv", index=False)
test_sample.head()


,Customer_Ref,CreditScore,Geography,Gender,Age,Balance,IsActiveMember,Exited
0,CUST-0001,596,Germany,Male,32,96709.07,0,0
1,CUST-0002,623,France,Male,43,0.00,1,0
2,CUST-0003,601,Spain,Female,44,0.00,0,0
3,CUST-0004,506,Germany,Male,59,119152.10,1,0
4,CUST-0005,560,Spain,Female,27,124995.98,1,0
